In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing
from sklearn.feature_selection import SelectKBest, f_regression
import time

## NumPy (Demonstration)

In [ ]:
# Create a large list and a large array (10 million numbers)
size = 10_000_000
py_list = list(range(size))
np_arr = np.array(range(size))

start_time = time.time()
# Python must check the type of every single integer in the loop
list_result = [x * 2 for x in py_list]
print(f"Python List Time: {time.time() - start_time:.4f} seconds")

start_time = time.time()
# NumPy knows the address and type (C-level efficiency) and vectorizes the operation
arr_result = np_arr * 2
print(f"NumPy Array Time: {time.time() - start_time:.4f} seconds")

Python List Time: 0.5048 seconds
NumPy Array Time: 0.0384 seconds


## Standard Data Format

In [ ]:
# Manually creating a dataset: 5 samples, 3 features
# each row represents a data sample and each column represents a feature
X = np.array([
    [25, 50000, 1],
    [32, 60000, 0],
    [40, 70000, 1],
    [29, 55000, 0],
    [35, 65000, 1]
])

# Target labels (labels) stored separately as a separate array
y = np.array([0, 1, 0, 1, 1])

print(f"X shape: {X.shape}") # (5, 3)
print(f"y shape: {y.shape}") # (5,)

X shape: (5, 3)
y shape: (5,)


## Transformers (Pre-Processing)

### Handling Categorical Data (One-Hot Encoding)


In [ ]:
# (2D array is required by sklearn; here there is only 1 categorical feature and 3 samples)
colors = np.array([["Red"], ["Blue"], ["Green"]])

# sparse_output=False: Returns a simple numpy array instead of a sparse matrix (easier to read)
encoder = OneHotEncoder(sparse_output=False)

# 'fit' learns the categories (Blue, Green, Red)
# 'transform' converts them to binary vectors
colors_encoded = encoder.fit_transform(colors)

print("Original Data:\n", colors)

print("\nOne-Hot Encoded Matrix:\n", colors_encoded)
# Note: Categories are sorted alphabetically by defaul
# Index 0 = Blue, Index 1 = Green, Index 2 = Red
# So, Red -> [0. 0. 1.] (3rd alphabetical category)

print("\nLearned Categories:", encoder.categories_)

# Inverse Transform
original_restored = encoder.inverse_transform(colors_encoded)
print("\nRestored Data:\n", original_restored)

Original Data:
 [['Red']
 ['Blue']
 ['Green']]

One-Hot Encoded Matrix:
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]

Learned Categories: [array(['Blue', 'Green', 'Red'], dtype='<U5')]

Restored Data:
 [['Red']
 ['Blue']
 ['Green']]


### Scaling

StandardScaler shifts data shifted so that the mean is 0 and the standard deviation is 1.

In [ ]:
# Splitting data first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train) # Learn and scale

# precision=2 rounds the display, suppress=True removes the 'e+01'
np.set_printoptions(precision=2, suppress=True)

# The average value for each feature in your training set
print(scaler.mean_)
# The relative scale (standard deviation) for each feature
print(scaler.scale_)

X_test_scaled = scaler.transform(X_test) # Scale only, no learning

print("\nOriginal Data:\n", X_train)
print("\nScaled Data:\n", X_train_scaled)

[   32.25 60000.       0.75]
[   5.72 7905.69    0.43]

Original Data:
 [[   35 65000     1]
 [   40 70000     1]
 [   25 50000     1]
 [   29 55000     0]]

Scaled Data:
 [[ 0.48  0.63  0.58]
 [ 1.36  1.26  0.58]
 [-1.27 -1.26  0.58]
 [-0.57 -0.63 -1.73]]


## Formatting Data

### The California Housing Dataset (Real-World Example)

In [ ]:
# Load the data
housing = fetch_california_housing()
X_house = pd.DataFrame(housing.data, columns=housing.feature_names)
y_house = housing.target

print("Features:", housing.feature_names)
print("Shape of X:", X_house.shape)
print("\nFirst 5 rows of X:\n", X_house.head())
print("\n\n")
print("Target:", housing.target_names)
print("Shape of y:", y_house.shape)
print("\nFirst 5 rows of y:\n", y_house[:5])

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Shape of X: (20640, 8)

First 5 rows of X:
    MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  



Target: ['MedHouseVal']
Shape of y: (20640,)

First 5 rows of y:
 [4.526 3.585 3.521 3.413 3.422]


### Splitting Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (16512, 8)
Shape of X_test: (4128, 8)
Shape of y_train: (16512,)
Shape of y_test: (4128,)


### Preparing Features for Model Training

In [ ]:
# Select top features (Optional)
selector = SelectKBest(f_regression, k=5)
X_train_fs = selector.fit_transform(X_train, y_train)
X_test_fs = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]
print ("Selected Features:", selected_features)

# Scale
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_fs)
X_test_final = scaler.transform(X_test_fs)

Selected Features: Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Latitude'], dtype='object')


## Train & Predict

In [ ]:
model = LinearRegression()
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

## Evaluation: How did we do?

In [ ]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared Score: {r2:.4f}")

# Using matplotlib we can also visualize to see if our predictions match the actual prices.
# import matplotlib.pyplot as plt

# plt.figure(figsize=(8, 6))
# plt.scatter(y_test, y_pred, alpha=0.3)
# plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--') # The "Perfect" line
# plt.xlabel("Actual Price")
# plt.ylabel("Predicted Price")
# plt.title("Actual vs Predicted")
# plt.show()

Mean Squared Error: 0.6383
R-squared Score: 0.5129


## Some Other Useful Stuff

### Pipelines

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Step 1: Scale
    ('selector', SelectKBest(k=5)),    # Step 2: Select Features
    ('regressor', LinearRegression())  # Step 3: Model
])

# Now we can treat the whole pipeline as a single model
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

### Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

# Instead of one split, we split the data 5 different ways (cv=5) for more reliable evaluation
scores = cross_val_score(pipeline, X_house, y_house, cv=5, scoring='r2')

print("Cross-Validation Scores:", scores)
print("Average Accuracy:", scores.mean())

Cross-Validation Scores: [0.55502496 0.44649628 0.53606573 0.5087421  0.67097857]
Average Accuracy: 0.5434615283078263


## **Common Scikit-learn Pitfalls**

**Data Leakage during Preprocessing**
- Applying transformations like scaling or imputation on the entire dataset before splitting lets information from the test set influence training.
- Always split data into training and testing sets first, then fit transformers only on training data.

**Ignoring Feature Scaling**
- Distance-based algorithms like SVMs and k-Nearest Neighbors, plus gradient-based methods like Neural Networks and Regularized Regression, are sensitive to feature magnitudes.
- For example, for distance-based algorithms, features with larger scales might dominate distance calculations.
- Use StandardScaler for standardization or MinMaxScaler for normalization with these algorithms.

**Shape Mismatches in Arrays**
- Scikit-learn transformers often expect 2D arrays with shape (n_samples, n_features), but single features come as 1D arrays with shape (n_samples,).
- This mismatch causes errors during transformation.
- Reshape single features using .reshape(-1, 1) to convert from 1D to 2D format.